In [ ]:
import matplotlib.pyplot as plt
from pymoo.util.nds.non_dominated_sorting import NonDominatedSorting
import numpy as np
import pandas as pd
import sys
import pickle

year = 2000
run_file_name = r"/rdata/ian/pico/paperRuns/global_run_table.pkl"
global_pf_file_name = r"/rdata/ian/pico/paperRuns/global_pf.pkl"

full_run_tab = pd.read_pickle(run_file_name)
global_pf = pd.read_pickle(global_pf_file_name)

global_pf = global_pf.sort_values(by="yield")



In [ ]:
selection_masks = [
    np.all([full_run_tab['algorithm'] == "pinsga2", full_run_tab['run'] == 0, full_run_tab['DM_range'] == "20to30"], axis=0),
    np.all([full_run_tab['algorithm'] == "nsga2", full_run_tab['run'] == 0], axis=0)]

labels = ["PI-NSGA-II", "NSGA-II"]

run_tabs = []
facecolors = ['none', 'green', 'purple', 'orange']
edgecolors = ['black', 'green', 'purple', 'orange']
markers = ['s', 'o', 'o', 'o']


In [ ]:
for (m, mask) in enumerate(selection_masks):
    run_tabs.append(full_run_tab.loc[mask, ['irr_total', 'yield', 'gen']])



In [ ]:
def filter_dom(run_tab):

    # Perform non-dominated sorting
    nds = NonDominatedSorting()

    minimize_pop = run_tab.copy()
    minimize_pop["yield"] = minimize_pop["yield"] * -1

    fronts = nds.do(minimize_pop.values, only_non_dominated_front=True)

    return run_tab.iloc[fronts,:].copy()
    

In [ ]:

gens = [200, 50]

rt_single_gen = [
    run_tab[run_tab["gen"] == gens[t]] for (t, run_tab) in enumerate(run_tabs)
]

# Plot the reference plot
for (d, run_tab) in enumerate(rt_single_gen):

    print(run_tab.shape)
    
    # Get the Pareto front
    pf = filter_dom(run_tab)

    # Plot the data (first column is f1, second column is f2)
    plt.scatter(pf.iloc[:,0], pf.iloc[:,1],
                label=labels[d],
                facecolors=facecolors[d],
                edgecolors=edgecolors[d],
                marker=markers[d])


# Plot the global pf
plt.plot(global_pf["irr_total"], global_pf["yield"],
            label="Near-true optima",
            color="red")


plt.xlabel("Irrigation (mm)")
plt.ylabel("Yield (kg/ha)")
plt.legend()


In [ ]:
run = 0

for (d, run_tab) in enumerate(run_tabs):

    for gen in range(1,max_gen):

        # Get the Pareto front
        pf = filter_dom(run_tab[run_tab["gen"] == gen])

        

